# 🔀 Unified AI API - Validation Tests

## Overview

Validate the Unified AI Wildcard API (`/unified-ai`) across different model providers and API patterns.

This notebook tests:
- **Access Contract Provisioning**: Deploy an access contract via Bicep to obtain API credentials
- **Model Discovery**: List available models via the Unified AI API using the access contract key
- **Azure OpenAI Pattern**: `/unified-ai/openai/deployments/{model}/chat/completions`
- **Inference Pattern (Foundry)**: `/unified-ai/models/chat/completions`
- **Gemini OpenAI Pattern**: `/unified-ai/v1beta/openai/chat/completions`
- **Deployment Discovery**: `GET /unified-ai/deployments` and `GET /unified-ai/deployments/{name}`
- **Authentication Modes**: API Key validation
- **Load Testing**: Concurrent requests with throttling visualization

> **Prerequisites:** An existing Citadel Governance Hub deployment with the Unified AI API enabled.

### 0️⃣ Initialize Notebook Variables

Configure the following variables according to your environment before running the notebook:

In [ ]:
import os
import sys, json, requests, time, shutil
sys.path.insert(1, '../shared')  # add the shared directory to the Python path
import utils
from apimtools import APIMClientTool

inference_api_version = "2024-05-01-preview"
openai_api_version = "2024-02-15-preview"

governance_hub_resource_group = "REPLACE"  ## specify the resource group name where the Governance Hub is located
location = "REPLACE"  ## specify the location of the Governance Hub

# Models to test (REPLACE based on your configured backends and deployed models)
openai_model = "gpt-4o"                      # Azure OpenAI model
foundry_inference_model = "Mistral-Large-3"  # AI Foundry inference model
gemini_model = "gemini-2.5-flash-lite"       # Gemini model (if Gemini backend configured)

# Set to True if Gemini backend is configured
test_gemini = False

# Access Contract configuration
contract_business_unit = "Testing"
contract_use_case_name = "UnifiedAI"
contract_environment = "DEV"

### 1️⃣ Verify Azure CLI and Connected Subscription

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

### 2️⃣ Initialize APIM Client Tool

Discover the Unified AI API and retrieve gateway configuration:

In [ ]:
try:
    apimClientTool = APIMClientTool(governance_hub_resource_group)
    apimClientTool.initialize()
    apimClientTool.discover_api("unified-ai")

    apim_resource_gateway_url = str(apimClientTool.apim_resource_gateway_url)
    azure_endpoint = str(apimClientTool.azure_endpoint)

    # Get supported models from the policy fragment
    supported_models = apimClientTool.get_policy_fragment_supported_models("set-backend-pools")
    utils.print_info(f"Supported models: {supported_models}")

    utils.print_ok(f"Testing tool initialized successfully!")
except Exception as e:
    utils.print_error(f"Error initializing APIM Client Tool: {e}")

### 3️⃣ Provision Access Contract

Deploy an access contract via Bicep to create an APIM product and subscription for testing the Unified AI API.
This grants API credentials scoped to the unified-ai-api.

In [ ]:
bicep_dir = "../bicep/infra/citadel-access-contracts"
template_file = os.path.join(bicep_dir, "main.bicep")

timestamp = time.strftime('%Y%m%d%H%M%S')
contract_name = f"unified-ai-test-{timestamp}"

# Create folder structure: contracts/[businessunit-usecase]/[environment]/
folder_name = f"{contract_business_unit.lower()}-{contract_use_case_name.lower()}"
environment_folder = contract_environment.lower()
contract_folder = os.path.join(bicep_dir, "contracts", folder_name, environment_folder)
os.makedirs(contract_folder, exist_ok=True)
utils.print_info(f"📁 Created folder: {contract_folder}")

# Build allowedModels from the notebook model variables
allowed_models_list = [openai_model, foundry_inference_model]
if test_gemini:
    allowed_models_list.append(gemini_model)
allowed_models_csv = ",".join(allowed_models_list)
utils.print_info(f"Allowed models for access contract: {allowed_models_csv}")

# Generate custom product policy with allowedModels scoped to the test models
product_policy = f'''<policies>
    <inbound>
        <base />
        <!-- Extract and validate model parameter from request -->
        <include-fragment fragment-id="set-llm-requested-model" />

        <!-- Setting allowed models to only the models defined in notebook parameters -->
        <set-variable name="allowedModels" value="{allowed_models_csv}" />

        <!-- Capacity management -->
        <llm-token-limit counter-key="@(context.Subscription.Id)" tokens-per-minute="5000" estimate-prompt-tokens="false" token-quota="100000" token-quota-period="Monthly" />

        <!-- Enable advanced response headers offered by set-response-headers fragment -->
        <set-variable name="enableResponseHeaders" value="@(true)" />
    </inbound>
    <backend>
        <base />
    </backend>
    <outbound>
        <base />
    </outbound>
    <on-error>
        <base />
    </on-error>
</policies>'''

# Write the custom policy file
policy_file_dest = os.path.join(contract_folder, "ai-product-policy.xml")
with open(policy_file_dest, 'w') as f:
    f.write(product_policy)
utils.print_ok(f"📋 Custom policy file created: {policy_file_dest}")

params_file = os.path.join(contract_folder, "main.bicepparam")

params_content = f"""using '../../../main.bicep'

// ============================================================================
// Unified AI API Test Contract - Generated from Notebook
// ============================================================================

param apim = {{
  subscriptionId: '{subscription_id}'
  resourceGroupName: '{governance_hub_resource_group}'
  name: '{apimClientTool.apim_resource_name}'
}}

param keyVault = {{
  subscriptionId: '00000000-0000-0000-0000-000000000000'
  resourceGroupName: 'placeholder'
  name: 'placeholder'
}}

param useTargetAzureKeyVault = false

param useCase = {{
  businessUnit: '{contract_business_unit}'
  useCaseName: '{contract_use_case_name}'
  environment: '{contract_environment}'
}}

param apiNameMapping = {{
  LLM: ['unified-ai-api']
}}

param services = [
  {{
    code: 'LLM'
    endpointSecretName: 'UNIFIED-AI-TEST-ENDPOINT'
    apiKeySecretName: 'UNIFIED-AI-TEST-KEY'
    policyXml: loadTextContent('ai-product-policy.xml')
  }}
]

param productTerms = 'Unified AI API test contract - generated from validation notebook'

// Azure AI Foundry Integration (disabled)
param useTargetFoundry = false

param foundry = {{
  subscriptionId: '00000000-0000-0000-0000-000000000000'
  resourceGroupName: 'placeholder'
  accountName: 'placeholder'
  projectName: 'placeholder'
}}
"""

with open(params_file, 'w') as f:
    f.write(params_content)
utils.print_ok(f"✅ Parameter file created: {params_file}")

# Deploy the access contract
product_id = f"LLM-{contract_business_unit}-{contract_use_case_name}-{contract_environment}"
utils.print_info(f"Deploying access contract: {contract_name} (Product: {product_id})...")

deployment_cmd = f"az deployment sub create --name {contract_name} --location {location} --template-file {template_file} --parameters {params_file}"
output = utils.run(deployment_cmd, f"Deployment '{contract_name}' succeeded", f"Deployment '{contract_name}' failed")

if output.success:
    utils.print_ok(f"✅ Access contract deployed! Product ID: {product_id}")
else:
    utils.print_error(f"❌ Access contract deployment failed!")

# Re-initialize APIM client to pick up the new subscription
apimClientTool.initialize()

### 4️⃣ Retrieve API Key from Access Contract

Get the subscription key created by the access contract deployment:

In [ ]:
subscription_name = f"{product_id}-SUB-01"

api_key = None
for sub in apimClientTool.apim_subscriptions:
    if subscription_name.lower() in sub.get('name', '').lower():
        api_key = sub.get('key')
        utils.print_ok(f"Found API key from subscription: {sub.get('name')}")
        break

if not api_key:
    raise Exception(f"No API key found for subscription '{subscription_name}'. Ensure the access contract was deployed successfully.")

# Build endpoint URLs for different API patterns
base_url = azure_endpoint.rstrip('/')
openai_url = f"{base_url}/unified-ai/openai/deployments/{openai_model}/chat/completions?api-version={openai_api_version}"
inference_url = f"{base_url}/unified-ai/models/chat/completions?api-version={inference_api_version}"
gemini_url = f"{base_url}/unified-ai/v1beta/openai/chat/completions"
deployments_url = f"{base_url}/unified-ai/deployments"

utils.print_info(f"OpenAI endpoint: {openai_url}")
utils.print_info(f"Inference endpoint: {inference_url}")
utils.print_info(f"Deployments endpoint: {deployments_url}")
utils.print_ok(f"API key and endpoints configured successfully!")

### 5️⃣ Discover Available Models

List all available models through the Unified AI API using the access contract key:

In [ ]:
utils.print_info("Listing all available deployments via the Unified AI API...")

try:
    response = requests.get(
        deployments_url,
        headers={'api-key': api_key},
        timeout=30
    )

    utils.print_response_code(response)

    if response.status_code == 200:
        data = json.loads(response.text)
        available_deployments = data.get("value", [])
        utils.print_ok(f"Found {len(available_deployments)} allowed models per access contract:")
        for d in available_deployments:
            name = d.get('name', 'unknown')
            model_format = d.get('properties', {}).get('model', {}).get('format', 'N/A')
            utils.print_info(f"  • {name} (format: {model_format})")
    else:
        utils.print_error(f"Error listing deployments: {response.text[:300]}")
        available_deployments = []
except Exception as e:
    utils.print_error(f"Request failed: {e}")
    available_deployments = []

# Cross-reference with policy-supported models
utils.print_info(f"\nGateway all supported models: {supported_models}")
deployment_names = [d.get('name', '') for d in available_deployments]
matched_models = [m for m in supported_models if m in deployment_names]
utils.print_ok(f"Models available for testing: {matched_models if matched_models else supported_models}")

---
## Test 1 — Azure OpenAI Pattern

Route through the OpenAI-compatible path: `POST /unified-ai/openai/deployments/{model}/chat/completions`

In [ ]:
utils.print_info(f"Testing Azure OpenAI pattern with model: {openai_model}")

messages = {
    "messages": [
        {"role": "system", "content": "You are a helpful assistant. Keep responses brief."},
        {"role": "user", "content": "What is 2+2? Answer in one word."}
    ]
}

try:
    response = requests.post(
        openai_url,
        headers={'api-key': api_key},
        json=messages,
        timeout=30
    )

    utils.print_response_code(response)

    if response.status_code == 200:
        data = json.loads(response.text)
        content = data.get("choices", [{}])[0].get("message", {}).get("content", "")
        utils.print_ok(f"💬 Response: {content}")
        utils.print_info(f"   Region: {response.headers.get('x-ms-region', 'N/A')}")
        utils.print_info(f"   Backend: {response.headers.get('UAIG-Backend', 'N/A')}")
        utils.print_info(f"   API Type: {response.headers.get('UAIG-API-Type', 'N/A')}")
    else:
        utils.print_error(f"Error: {response.text[:300]}")
except Exception as e:
    utils.print_error(f"Request failed: {e}")

## Test 2 — Inference Pattern (Foundry Models)

Route through the inference path: `POST /unified-ai/models/chat/completions`

In [ ]:
utils.print_info(f"Testing Foundry Inference with model: {foundry_inference_model}")

inference_messages = {
    "model": foundry_inference_model,
    "messages": [
        {"role": "system", "content": "You are a helpful assistant. Keep responses brief."},
        {"role": "user", "content": "What is the capital of France? Answer in one word."}
    ]
}

try:
    response = requests.post(
        inference_url,
        headers={'api-key': api_key},
        json=inference_messages,
        timeout=60
    )

    utils.print_response_code(response)

    if response.status_code == 200:
        data = json.loads(response.text)
        content = data.get("choices", [{}])[0].get("message", {}).get("content", "")
        utils.print_ok(f"💬 Response: {content[:200]}")
        utils.print_info(f"   Backend: {response.headers.get('UAIG-Backend', 'N/A')}")
        utils.print_info(f"   API Type: {response.headers.get('UAIG-API-Type', 'N/A')}")
    else:
        utils.print_error(f"Error: {response.text[:300]}")
except Exception as e:
    utils.print_error(f"Request failed: {e}")

## Test 3 — Deployment Discovery

Validate deployment listing through `GET /unified-ai/deployments`

In [ ]:
# List all deployments
utils.print_info("Listing all available deployments...")

try:
    response = requests.get(
        deployments_url,
        headers={'api-key': api_key},
        timeout=30
    )

    utils.print_response_code(response)

    if response.status_code == 200:
        data = json.loads(response.text)
        deployments = data.get("value", [])
        utils.print_ok(f"Found {len(deployments)} deployments:")
        for d in deployments:
            name = d.get('name', 'unknown')
            model_format = d.get('properties', {}).get('model', {}).get('format', 'N/A')
            utils.print_info(f"  • {name} (format: {model_format})")
    else:
        utils.print_error(f"Error: {response.text[:300]}")
except Exception as e:
    utils.print_error(f"Request failed: {e}")

In [ ]:
# Get specific deployment (should return 200)
utils.print_info(f"Getting deployment: {openai_model}")

try:
    response = requests.get(
        f"{deployments_url}/{openai_model}",
        headers={'api-key': api_key},
        timeout=30
    )

    utils.print_response_code(response)
    if response.status_code == 200:
        data = json.loads(response.text)
        utils.print_ok(f"Found deployment: {data.get('name', 'unknown')}")
    else:
        utils.print_error(f"Unexpected: {response.text[:200]}")
except Exception as e:
    utils.print_error(f"Request failed: {e}")

# Get non-existent deployment (should return 404)
utils.print_info(f"Getting non-existent deployment: nonexistent-model-xyz")

try:
    response = requests.get(
        f"{deployments_url}/nonexistent-model-xyz",
        headers={'api-key': api_key},
        timeout=30
    )

    if response.status_code == 404:
        utils.print_ok(f"Correctly returned 404 for non-existent model")
    else:
        utils.print_error(f"Unexpected status {response.status_code}: {response.text[:200]}")
except Exception as e:
    utils.print_error(f"Request failed: {e}")

## Test 4 — Gemini OpenAI Pattern (if configured)

Route through the Gemini-compatible path: `POST /unified-ai/v1beta/openai/chat/completions`

In [ ]:
if test_gemini:
    utils.print_info(f"Testing Gemini OpenAI pattern with model: {gemini_model}")

    gemini_messages = {
        "model": gemini_model,
        "messages": [
            {"role": "user", "content": "What is the speed of light? Answer briefly."}
        ]
    }

    try:
        response = requests.post(
            gemini_url,
            headers={'api-key': api_key},
            json=gemini_messages,
            timeout=30
        )

        utils.print_response_code(response)

        if response.status_code == 200:
            data = json.loads(response.text)
            content = data.get("choices", [{}])[0].get("message", {}).get("content", "")
            utils.print_ok(f"💬 Gemini Response: {content[:200]}")
            utils.print_info(f"   Backend: {response.headers.get('UAIG-Backend', 'N/A')}")
            utils.print_info(f"   API Type: {response.headers.get('UAIG-API-Type', 'N/A')}")
        else:
            utils.print_error(f"Error: {response.text[:300]}")
    except Exception as e:
        utils.print_error(f"Request failed: {e}")
else:
    utils.print_info("Skipping Gemini test (test_gemini = False)")

## Test 5 — API Key Authentication

Validate that API Key authentication works and missing keys are rejected:

In [ ]:
# Test: Valid API Key -> should succeed
utils.print_info("Testing with valid API key...")

response = requests.post(
    openai_url,
    headers={'api-key': api_key},
    json={"messages": [{"role": "user", "content": "Hi"}]},
    timeout=30
)
if response.status_code == 200:
    utils.print_ok(f"✅ Valid API key: {response.status_code} (expected 200)")
else:
    utils.print_error(f"❌ Valid API key: {response.status_code} (expected 200)")

# Test: Missing API Key -> should return 401
utils.print_info("Testing without API key...")

response = requests.post(
    openai_url,
    headers={},
    json={"messages": [{"role": "user", "content": "Hi"}]},
    timeout=30
)
if response.status_code == 401:
    utils.print_ok(f"✅ Missing API key: {response.status_code} (expected 401)")
else:
    utils.print_error(f"❌ Missing API key: {response.status_code} (expected 401)")

## Test 6 — Load Test & Throttling

Send multiple requests over 30 seconds to observe rate limiting behavior:

In [ ]:
test_duration = 30
api_runs = []

test_model = openai_model
test_url = openai_url
utils.print_info(f"Load testing with model: {test_model} for {test_duration} seconds")

messages = {
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Count from 1 to 5."}
    ]
}

start_time = time.time()
run_count = 0

while (time.time() - start_time) < test_duration:
    run_count += 1
    call_start = time.time()

    try:
        response = requests.post(
            test_url,
            headers={'api-key': api_key},
            json=messages,
            timeout=30
        )

        elapsed = time.time() - start_time
        total_tokens = 0
        if response.status_code == 200:
            data = json.loads(response.text)
            total_tokens = data.get("usage", {}).get("total_tokens", 0)

        api_runs.append((call_start, total_tokens, response.status_code, elapsed))
    except Exception as e:
        api_runs.append((call_start, 0, 500, time.time() - start_time))

    time.sleep(0.2)

success = sum(1 for r in api_runs if r[2] == 200)
throttled = sum(1 for r in api_runs if r[2] == 429)
errors = sum(1 for r in api_runs if r[2] not in [200, 429])

utils.print_ok(f"Load test complete: {len(api_runs)} calls")
utils.print_info(f"  ✅ Success: {success} | ⛔ Throttled: {throttled} | ❌ Errors: {errors}")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

if api_runs:
    fig, ax = plt.subplots(1, 1, figsize=(14, 5))

    times = [r[3] for r in api_runs]
    tokens = [r[1] for r in api_runs]
    status_codes = [r[2] for r in api_runs]

    colors = [
        'tab:green' if code == 200
        else 'tab:red' if code == 429
        else 'tab:orange'
        for code in status_codes
    ]

    ax.bar(times, tokens, color=colors, width=0.3, alpha=0.7)

    throttled_times = [t for t, code in zip(times, status_codes) if code == 429]
    if throttled_times:
        max_tokens = max(tokens) if tokens else 1
        ax.scatter(throttled_times, [max_tokens * 0.05] * len(throttled_times),
                  marker='x', s=50, color='darkred', zorder=5)

    total_tokens = sum(tokens)
    stats_text = f"Total: {len(api_runs)} calls | Success: {success} | Throttled: {throttled} | Tokens: {total_tokens}"
    ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, fontsize=9,
           verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    ax.set_title('Unified AI API - Load Test Results', fontsize=12, fontweight='bold')
    ax.set_xlabel('Time (seconds)')
    ax.set_ylabel('Tokens per call')

    legend_items = [
        Patch(facecolor='tab:green', alpha=0.7, label='Success (200)'),
        Patch(facecolor='tab:red', alpha=0.7, label='Throttled (429)'),
        Patch(facecolor='tab:orange', alpha=0.7, label='Error'),
    ]
    ax.legend(handles=legend_items, loc='upper right')

    plt.tight_layout()
    plt.show()
else:
    print('No data to visualize. Run the load test first.')

## Summary

| Test | Pattern | Expected |
|------|---------|----------|
| Access Contract | Deploy Bicep access contract | Product + subscription created |
| Model Discovery | `GET /unified-ai/deployments` | Available model list |
| Azure OpenAI | `POST /unified-ai/openai/deployments/{model}/chat/completions` | 200 |
| Foundry Inference | `POST /unified-ai/models/chat/completions` (body: model) | 200 |
| List Deployments | `GET /unified-ai/deployments` | 200 with model list |
| Get Deployment | `GET /unified-ai/deployments/{name}` | 200 or 404 |
| Gemini | `POST /unified-ai/v1beta/openai/chat/completions` | 200 (if configured) |
| Valid API Key | Request with api-key header | 200 |
| Missing API Key | Request without api-key header | 401 |
| Load Test | 30s burst requests | Mix of 200/429 |

---
### 🧹 Cleanup

Remove the test access contract from APIM created during this notebook session:

In [ ]:
# Set to True to delete the access contract created in this session
cleanup_enabled = True

if cleanup_enabled:
    utils.print_info(f"Deleting access contract product: {product_id}...")

    prod_cmd = f"az apim product delete --resource-group {governance_hub_resource_group} --service-name {apimClientTool.apim_resource_name} --product-id {product_id} --delete-subscriptions true --yes"
    utils.run(prod_cmd, f"Deleted product {product_id}", f"Failed to delete product {product_id}")

    utils.print_ok("Cleanup completed!")
else:
    utils.print_info("Cleanup is disabled. Set cleanup_enabled = True to remove test resources.")